# Expanded review collection: 5,000 games × up to 500 reviews

Run all cells from the project root or notebooks/. Everything is saved under **data/reviews_extanded/**; the original review cache and game list are untouched. `500` is a maximum per game, not a guarantee. At most 2.5 million reviews; 25,000 review requests at a two-second interval take roughly 14 hours plus network time.

Popularity follows the existing project approach: SteamSpy's estimated-owner ranking, not concurrent players. Ranking pages are cached as a snapshot, with game names and IDs in games.json. Reviews use Steam's most-helpful sort with `day_range=0` (all time), default review-quality ranking, English language, both sentiments, Steam purchasers, and default off-topic filtering. This is a helpfulness-ranked sample, not random sampling.

Each game's CSV has the same columns as the original cache. It is replaced atomically after each page; cursors are saved separately afterward, so restarting can safely replay a page without duplicating review IDs. Completed and exhausted games are skipped. Failed games remain eligible on rerun. Keep the same configuration when resuming; use another directory for a different sample.

Standard library only. Existing analysis notebooks still read the original cache; this notebook does not redirect them.

Sources: [SteamSpy API](https://steamspy.com/api.php) · [Steam review API](https://partner.steamgames.com/doc/webapi/IUserReviewsService)

In [1]:
import csv
import json
import sys
import time
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.csv_io import CACHE_FIELDS
from src.steam_reviews_client import get_game_reviews

N_GAMES, REVIEWS_PER_GAME = 5000, 500
OUT = ROOT / "data" / "reviews_extanded"
STATE = OUT / "progress"
STATE.mkdir(parents=True, exist_ok=True)
STEAMSPY_DELAY, REVIEW_DELAY = 5, 0.5
last_request = 0


def save_json(path, value):
    temporary = path.with_suffix(".tmp")
    temporary.write_text(json.dumps(value, indent=2) + "\n", encoding="utf-8")
    temporary.replace(path)


def request(call, delay):
    global last_request
    for attempt in range(4):
        time.sleep(max(0, delay - (time.monotonic() - last_request)))
        try:
            return call()
        except (HTTPError, URLError, RuntimeError, TimeoutError, ValueError) as exc:
            if attempt == 3:
                raise
            retry_after = exc.headers.get("Retry-After", "") if isinstance(exc, HTTPError) else ""
            wait = max(60 * (attempt + 1), int(retry_after) if retry_after.isdigit() else 0)
            print(f"{exc}; retrying in {wait}s", flush=True)
            time.sleep(wait)
        finally:
            last_request = time.monotonic()


def steamspy_page(page):
    req = Request(f"https://steamspy.com/api.php?request=all&page={page}",
                  headers={"User-Agent": "GameRecommender/1.0"})
    with urlopen(req, timeout=60) as response:
        data = json.load(response)
    if not isinstance(data, dict) or any(not isinstance(g, dict) or "appid" not in g for g in data.values()):
        raise ValueError("Unexpected SteamSpy page")
    return data


In [2]:
games, page = {}, 0
while len(games) < N_GAMES:
    cache = OUT / f"steamspy_page_{page}.json"
    if cache.exists():
        data = json.loads(cache.read_text(encoding="utf-8"))
    else:
        data = request(lambda: steamspy_page(page), STEAMSPY_DELAY)
        save_json(cache, data)
    before = len(games)
    for game in data.values():
        appid = int(game["appid"])
        if 0 < appid <= 2**32 - 1 and appid != 999999:
            games.setdefault(appid, game)
    print(f"Ranking page {page}: {len(games):,} usable unique games", flush=True)
    if len(games) == before:
        break
    page += 1

games = list(games.values())[:N_GAMES]
app_ids = [int(game["appid"]) for game in games]
save_json(OUT / "games.json", games)
print(f"Selected {len(games):,} / {N_GAMES:,} requested games")


Ranking page 0: 1,000 usable unique games
Ranking page 1: 2,000 usable unique games
Ranking page 2: 3,000 usable unique games
Ranking page 3: 3,998 usable unique games
Ranking page 4: 4,996 usable unique games
Ranking page 5: 5,992 usable unique games
Selected 5,000 / 5,000 requested games


In [3]:
def collect(appid):
    path, checkpoint = OUT / f"{appid}.csv", STATE / f"{appid}.json"
    rows = {}
    if path.exists():
        with path.open(encoding="utf-8", newline="") as file:
            rows = {row["recommendationid"]: row for row in csv.DictReader(file)}
    state = json.loads(checkpoint.read_text()) if checkpoint.exists() else {}
    if len(rows) >= REVIEWS_PER_GAME or state.get("exhausted"):
        return len(rows)
    cursor, visited = state.get("cursor", "*"), set()
    while len(rows) < REVIEWS_PER_GAME:
        if cursor in visited:
            raise RuntimeError("Repeated review cursor; rerun to retry this game")
        visited.add(cursor)
        result = request(lambda: get_game_reviews(
            appid, filter=0, day_range=0, cursor=cursor, num_per_page=min(100, REVIEWS_PER_GAME - len(rows)),
        ), REVIEW_DELAY)
        payload = result.get("response")
        if not isinstance(payload, dict) or not isinstance(payload.get("reviews"), list):
            raise RuntimeError("Unexpected review response; cursor not advanced")
        batch = payload["reviews"]
        for review in batch:
            review_id = str(review["recommendationid"])
            rows[review_id] = {
                "appid": appid, "recommendationid": review_id,
                "author_id": review.get("author", {}).get("steamid", ""),
                "review": review.get("review", ""),
                "weighted_vote_score": review.get("weighted_vote_score", 0),
                "votes_up": review.get("votes_up", 0), "voted_up": review.get("voted_up", ""),
            }
            if len(rows) >= REVIEWS_PER_GAME:
                break
        temporary = path.with_suffix(".tmp")
        with temporary.open("w", encoding="utf-8", newline="") as file:
            writer = csv.DictWriter(file, fieldnames=CACHE_FIELDS)
            writer.writeheader()
            writer.writerows(rows.values())
        temporary.replace(path)
        next_cursor = payload.get("cursor")
        matching = payload.get("total_matching")
        exhausted = not batch or (type(matching) is int and matching <= len(rows))
        if not exhausted and len(rows) < REVIEWS_PER_GAME and not next_cursor:
            raise RuntimeError("Missing next cursor; saved reviews but not progress")
        save_json(checkpoint, {"cursor": next_cursor, "exhausted": exhausted, "reviews": len(rows)})
        if exhausted:
            break
        cursor = next_cursor
    return len(rows)


In [4]:
failures, total = [], 0
for index, appid in enumerate(app_ids, 1):
    try:
        count = collect(appid)
        total += count
        print(f"[{index}/{len(app_ids)}] {appid}: {count} reviews", flush=True)
    except (RuntimeError, OSError, ValueError, KeyError) as exc:
        failures.append({"appid": appid, "error": str(exc)})
        save_json(OUT / "failures.json", failures)
        print(f"[{index}/{len(app_ids)}] {appid}: FAILED — {exc}", flush=True)
save_json(OUT / "failures.json", failures)
print(f"Finished: {total:,} reviews in successful/skipped games; {len(failures)} failures")


[1/5000] 730: 500 reviews
[2/5000] 1172470: 500 reviews
[3/5000] 578080: 500 reviews
[4/5000] 1623730: 500 reviews
[5/5000] 440: 500 reviews
[6/5000] 1938090: 500 reviews
[7/5000] 1063730: 500 reviews
[8/5000] 2358720: 500 reviews
[9/5000] 271590: 500 reviews
[10/5000] 550: 500 reviews
[11/5000] 304930: 500 reviews
[12/5000] 1599340: 500 reviews
[13/5000] 236390: 500 reviews
[14/5000] 2246340: 500 reviews
[15/5000] 553850: 500 reviews
[16/5000] 230410: 500 reviews
[17/5000] 1245620: 500 reviews
[18/5000] 105600: 500 reviews
[19/5000] 2694490: 500 reviews
[20/5000] 431960: 500 reviews
[21/5000] 1086940: 500 reviews
[22/5000] 291550: 500 reviews
[23/5000] 4000: 500 reviews
[24/5000] 1085660: 500 reviews
[25/5000] 359550: 500 reviews
[26/5000] 252490: 500 reviews
[27/5000] 552990: 500 reviews
[28/5000] 346110: 500 reviews
[29/5000] 413150: 500 reviews
[30/5000] 242760: 500 reviews
[31/5000] 1491000: 369 reviews
[32/5000] 238960: 500 reviews
[33/5000] 218620: 500 reviews
[34/5000] 291480: 

KeyboardInterrupt: 